# SOLiD-PINN v0.9.0e — Robustness and mechanism competition
CPU · Run all · No Drive · No PAT. 924 planned fits. Known IID or AR(1) noise at 5 mV; the covariance is supplied, not inferred. All data synthetic. A selected mechanism is not proof of structural identifiability.
## 1. Pinned source and regression tests


In [ ]:
from pathlib import Path
from google.colab import files
import subprocess,sys,os,tempfile,json,hashlib,math
REV="8f7e6a98d1c33ec6b347eeaa4190899fd82b17f8"
repo=Path(tempfile.mkdtemp(prefix="solid-v090e-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(repo)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=repo,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()==REV
hashes={"robust_solver.py":"203bbf7c41cb272155a830036bb55db34c93ec18ed097e5b8049d57ae043ff29","robust_mechanisms.py":"7b06e71469c23bf3d7c62f24cf7b939fbe7b6f8121b0b6bf1fa4cb6eff85228a","test_robust_mechanisms.py":"02e2608cf3e375825f77a5efad908023898075fd6d10fb5a819eb2dfc6b15988","coupled_bridge.py":"87f474918e731710193a4b01ee2075d5cc6a157e12fc1c9e1fdcf3779b37d3e3","adaptive_coupled.py":"7f46443b8f09199b9c96f12197555408e81783c8f5733cd19fcec4b3395c7256","hybrid_p2d.py":"20aaeede052a29a2b65b1cd322b9fade3b4094fd4e1564d15787d7e240c51f18"}
for name,digest in hashes.items(): assert hashlib.sha256((repo/"scripts"/name).read_bytes()).hexdigest()==digest
env=dict(os.environ,OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_robust_mechanisms.py"],cwd=repo,env=env,check=True)


## 2. Execute benchmark
29 null calibration cases per noise regime; 24 held-out nulls and four cases per each of six alternative truths per regime. Six models per case. Fine reference 64×96/16, training 32×48/4. Checkpoint after each case. If interrupted, run the remaining audit/export cells manually; do not restart benchmark or disconnect before downloading. Checkpoint is not persistent across runtime resets.


In [ ]:
root=Path("/content/SOLiD-PINN/v090e/handoff");root.mkdir(parents=True,exist_ok=True)
out=root/"checkpoint.json"
# Atomic checkpoint is preserved in the current runtime after every completed case.
try:
    subprocess.run([sys.executable,"-u","scripts/robust_mechanisms.py","--out",str(out)],cwd=repo,env=env,check=True)
except (KeyboardInterrupt,subprocess.CalledProcessError) as e:
    print("Interrupted or failed; exporting available checkpoint:",repr(e))


## 3. Audit choices and print counts


In [ ]:
d=json.loads(out.read_text())
methods=("calibrated","linear","rbf","diffusion","kinetic","series")
truths=("null","rest","peak","weak_rest","diffusion","kinetic","series")
cal={(r,"null",s) for r in (0.,.8) for s in range(3000,3029)}
test={(r,t,s) for r in (0.,.8) for t in truths for s in (range(4000,4024) if t=="null" else range(4100,4104))}
for name,expected in (("calibration",cal),("test",test)):
    keys=[(x["rho"],x["truth"],x["seed"]) for x in d[name]]
    assert len(keys)==len(set(keys)) and set(keys)<=expected
    if d["complete"]: assert set(keys)==expected
    for x in d[name]:
        if not x.get("fit_gate_passed"): continue
        assert x["truth_gate_passed"] and max(x["mesh_delta_mv"].values())<=1
        assert set(x["models"])==set(methods)
        assert all(m["success"] and m["physical_checks_passed"] for m in x["models"].values())
        s=x["validation_scores"];best=min(methods[1:],key=s.get)
        assert math.isclose(x["gain"],s["calibrated"]-s[best],abs_tol=1e-12)
        assert x["raw"]==(best if x["gain"]>0 else "calibrated")
        if name=="test": assert x["gated"]==(best if x["gain"]>d["thresholds"][str(x["rho"])] else "calibrated")
for r,q in d["thresholds"].items():
    a=sorted(x["gain"] for x in d["calibration"] if x["rho"]==float(r))
    assert len(a)==29 and q==max(0.,a[26])
d.update(source_revision=REV,source_hashes=hashes,targeted_tests_passed=5)
print("Complete:",d["complete"],"Calibration:",len(d["calibration"]),"Test:",len(d["test"]))
for r in (0.,.8):
    for t in truths:
        rows=[x for x in d["test"] if x["rho"]==r and x["truth"]==t and "gated" in x]
        if rows:
            print(r,t,{m:sum(x["gated"]==m for x in rows) for m in methods},len(rows))


## 4. Export complete or partial results


In [ ]:
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
dest=root/f"SOLiD-PINN_v090e_colab_{sha[:20]}.json"
tmp=dest.with_suffix(".part");tmp.write_bytes(raw);tmp.replace(dest)
assert dest.read_bytes()==raw
print("LOCAL FILE VERIFIED",dest.name,sha)
try: files.download(str(dest))
except Exception as e: print("Download manually from Files panel:",str(dest),str(e))
